# D090 — Classes & Objects

**E-commerce story:** QuickCart handles thousands of products. Each product keeps related data and behaviour together: an ID, name, price and stock, plus operations such as applying a discount or updating a price.

## Classes and objects

A **class** is a blueprint for creating a particular kind of value. It defines the data each value stores (**attributes**) and the operations it performs (**methods**).

An **object** is one independent instance created from a class. A smartphone and a laptop can both be `Product` objects, but each keeps its own attribute values.
- `__init__` is the **constructor initializer**. Python calls it automatically after creating an object.
- `self` means “this object”. It must be the first parameter of an instance method, but is not supplied explicitly when calling the method.
- An **instance method** reads or changes one object's state and therefore receives `self`.
- A **static method** belongs logically to the class but needs neither object state nor class state. Mark it with `@staticmethod`.
- Python has no Java/C++-style constructor overloading by signature. A later `__init__` replaces an earlier one. Use default values, keyword arguments, or named alternative constructors instead.
- `__str__` gives a friendly, user-facing description; `str(obj)` and `print(obj)` use it.
- `__repr__` gives an unambiguous developer-facing representation; ideally, `eval(repr(obj))` can rebuild an equivalent object when it is safe and practical.

## 1. Defining a class and creating objects

The `class` keyword defines a class. Calling the class, as in `Product(...)`, creates an object. `Product` is the blueprint; `phone` and `laptop` are separate objects created from it. Changing one object does not change the other.

In [ ]:
class Product:
    def __init__(self, product_id, name, price):
        self.product_id = product_id
        self.name = name
        self.price = price


phone = Product("P101", "Smartphone", 29999.0)
laptop = Product("P102", "Laptop", 64999.0)

print(phone.name, phone.price)
print(laptop.name, laptop.price)
print(phone is laptop)  # Two independent objects

## 2. Constructor, instance methods, and `self`

`__init__` initializes a new object. Python calls it automatically when `Product(...)` is executed. An assignment such as `self.price = price` stores the received value in that particular object.

`self` means the object on which the method is currently working. It is the first parameter of an instance method. A function inside a class is a **member function** or **method**; when it receives `self`, it can read or change that object's attributes.

Calling `phone.discounted_price(10)` is equivalent to `Product.discounted_price(phone, 10)`. In the normal call, Python supplies `phone` as `self` automatically.

In [ ]:
class Product:
    def __init__(self, product_id, name, price):
        self.product_id = product_id
        self.name = name
        self.price = price

    def discounted_price(self, discount_percent):
        return round(self.price * (1 - discount_percent / 100), 2)

    def update_price(self, new_price):
        self.price = new_price


phone = Product("P101", "Smartphone", 29999.0)

print(phone.discounted_price(10))
print(Product.discounted_price(phone, 10))  # Same call, explicit self

phone.update_price(28999.0)
print(phone.price)

## 3. Static methods

A **static method** is placed inside a class because it is closely related to that class, but it does not read or change an object's attributes. It therefore has no `self` parameter and is marked with `@staticmethod`.

Checking a product-ID format belongs with `Product`, but does not require a particular product object. Call it through the class: `Product.is_valid_product_id(...)`.

In [ ]:
class Product:
    def __init__(self, product_id, name, price):
        if not Product.is_valid_product_id(product_id):
            raise ValueError("product_id must look like P101")
        self.product_id = product_id
        self.name = name
        self.price = price

    @staticmethod
    def is_valid_product_id(product_id):
        return (
            isinstance(product_id, str)
            and len(product_id) >= 2
            and product_id.startswith("P")
            and product_id[1:].isdigit()
        )


print(Product.is_valid_product_id("P101"))
print(Product.is_valid_product_id("ITEM-1"))

phone = Product("P101", "Smartphone", 29999.0)

## 4. Constructor options and keyword arguments

Constructor parameters can have default values. Here, stock starts at `0` and category starts as `"General"` unless the caller supplies other values.

Positional arguments must follow the constructor's parameter order. **Keyword arguments** use parameter names, make long calls easier to read, and may be supplied in a different order.

In [ ]:
class Product:
    def __init__(self, product_id, name, price, stock=0, category="General"):
        self.product_id = product_id
        self.name = name
        self.price = float(price)
        self.stock = stock
        self.category = category


# Positional arguments; defaults are used
mouse = Product("P201", "Wireless Mouse", 1499)

# Keyword arguments: readable and order-independent
keyboard = Product(
    name="Mechanical Keyboard",
    product_id="P202",
    category="Electronics",
    price=3499,
    stock=25,
)

print(mouse.__dict__)
print(keyboard.__dict__)

## 5. “Overloading” constructors in Python

Languages such as Java allow several constructors with different parameter lists. Python does **not** overload methods by signature. Defining two methods named `__init__` does not create overloads—the second definition replaces the first.

Python supports different construction styles with default values, keyword arguments, or a named alternative constructor such as `from_catalog_row`.

`from_catalog_row` converts an imported catalogue row and then calls the regular constructor. `@classmethod` supplies the class as `cls`; `cls(...)` creates and returns the new object. This example does not use inheritance.

In [ ]:
class Product:
    def __init__(self, product_id, name, price, stock=0, category="General"):
        self.product_id = product_id
        self.name = name
        self.price = float(price)
        self.stock = stock
        self.category = category

    @classmethod
    def from_catalog_row(cls, row):
        """Named alternative constructor for imported e-commerce data."""
        product_id, name, price, stock, category = row.split(",")
        return cls(
            product_id=product_id,
            name=name,
            price=float(price),
            stock=int(stock),
            category=category,
        )


book = Product("P301", "Python Handbook", 799)  # Regular constructor
headphones = Product.from_catalog_row(
    "P302,Noise Cancelling Headphones,8999,12,Electronics"
)  # Alternative constructor

print(book.__dict__)
print(headphones.__dict__)

## 6. `__str__` and `__repr__`

Without these methods, printing an object usually shows its class and memory address rather than useful business information.

- `__str__` returns readable output for people. `str(product)` and `print(product)` use it.
- `__repr__` returns precise output for developers and debugging. A notebook uses it when an object is the final expression in a cell.

A clean `repr` often resembles the constructor call. The `!r` conversion applies `repr()` to each attribute, preserving quotation marks and escaped characters.

The representation below is valid Python code. Because `Product` is available in the current scope, evaluating it calls the constructor and rebuilds an equivalent object.

In [ ]:
class Product:
    def __init__(self, product_id, name, price, stock=0, category="General"):
        self.product_id = product_id
        self.name = name
        self.price = float(price)
        self.stock = stock
        self.category = category

    def __str__(self):
        return f"{self.name} — ₹{self.price:,.2f} ({self.stock} in stock)"

    def __repr__(self):
        return (
            f"Product(product_id={self.product_id!r}, name={self.name!r}, "
            f"price={self.price!r}, stock={self.stock!r}, "
            f"category={self.category!r})"
        )


product = Product(
    product_id="P401",
    name="27-inch Monitor",
    price=18999,
    stock=8,
    category="Electronics",
)

print(str(product))       # Friendly output
print(repr(product))      # Developer output
product                   # Notebooks display repr(object)

### Rebuild an object from `repr`

`repr(product)` produces a constructor expression containing all the data required to recreate the product. Evaluating that expression creates a different object with the same attribute values.

`eval` executes text as Python code. It is used here only to demonstrate the `repr` contract—**never evaluate untrusted input**. For real data exchange, use JSON or another serialization format.

In [ ]:
product_code = repr(product)
rebuilt_product = eval(product_code, {"Product": Product})

print("Code:", product_code)
print("Rebuilt:", rebuilt_product)
print("Different objects:", rebuilt_product is not product)
print("Same data:", rebuilt_product.__dict__ == product.__dict__)

## Quick recap

```python
product = Product(...)                 # construct an object
product.discounted_price(10)           # call an instance method; self is automatic
Product.is_valid_product_id("P101")   # call a static method through the class
str(product)                           # friendly text
repr(product)                          # precise developer text
```

**Key distinction:** An instance method needs `self` because it works with one object's state. A static method performs class-related work that requires no object state.